In [95]:
import pandas as pd
import numpy as np

full_pumpkins = pd.read_csv('../../../2-Regression/data/US-pumpkins.csv')

full_pumpkins.head()


,City Name,Type,Package,Variety,Sub Variety,Grade,Date,Low Price,High Price,Mostly Low,...,Unit of Sale,Quality,Condition,Appearance,Storage,Crop,Repack,Trans Mode,Unnamed: 24,Unnamed: 25
0,BALTIMORE,NaN,24 inch bins,NaN,NaN,NaN,4/29/17,270.0,280.0,270.0,...,NaN,NaN,NaN,NaN,NaN,NaN,E,NaN,NaN,NaN
1,BALTIMORE,NaN,24 inch bins,NaN,NaN,NaN,5/6/17,270.0,280.0,270.0,...,NaN,NaN,NaN,NaN,NaN,NaN,E,NaN,NaN,NaN
2,BALTIMORE,NaN,24 inch bins,HOWDEN TYPE,NaN,NaN,9/24/16,160.0,160.0,160.0,...,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN
3,BALTIMORE,NaN,24 inch bins,HOWDEN TYPE,NaN,NaN,9/24/16,160.0,160.0,160.0,...,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN
4,BALTIMORE,NaN,24 inch bins,HOWDEN TYPE,NaN,NaN,11/5/16,90.0,100.0,90.0,...,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN


In [96]:
columns_to_select = ['City Name', 'Package', 'Variety', 'Origin', 'Item Size', 'Color']
pumpkins = full_pumpkins.loc[:, columns_to_select]

pumpkins = pumpkins.dropna()

pumpkins.info()

<class 'pandas.DataFrame'>
Index: 991 entries, 2 to 1698
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   City Name  991 non-null    str  
 1   Package    991 non-null    str  
 2   Variety    991 non-null    str  
 3   Origin     991 non-null    str  
 4   Item Size  991 non-null    str  
 5   Color      991 non-null    str  
dtypes: str(6)
memory usage: 54.2 KB


In [97]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder

item_size_categories = [['sml', 'med', 'med-lge', 'lge', 'xlge', 'jbo', 'exjbo']]
ordinal_features = ['Item Size']
ordinal_encoder = OrdinalEncoder(categories=item_size_categories)



categorical_features = ['City Name', 'Package', 'Variety', 'Origin']
categorical_encoder = OneHotEncoder(sparse_output=False)


ct = ColumnTransformer(transformers=[
    ('ord', ordinal_encoder, ordinal_features),
    ('cat', categorical_encoder, categorical_features)
    ])

ct.set_output(transform='pandas')
encoded_features = ct.fit_transform(pumpkins)

label_encoder = LabelEncoder()
encoded_label = label_encoder.fit_transform(pumpkins['Color'])

encoded_pumpkins = encoded_features.assign(Color=encoded_label)
print(encoded_pumpkins.columns)
encoded_pumpkins.head()

Index(['ord__Item Size', 'cat__City Name_ATLANTA', 'cat__City Name_BALTIMORE',
       'cat__City Name_BOSTON', 'cat__City Name_CHICAGO',
       'cat__City Name_COLUMBIA', 'cat__City Name_DALLAS',
       'cat__City Name_DETROIT', 'cat__City Name_LOS ANGELES',
       'cat__City Name_MIAMI', 'cat__City Name_NEW YORK',
       'cat__City Name_PHILADELPHIA', 'cat__City Name_SAN FRANCISCO',
       'cat__City Name_ST. LOUIS', 'cat__Package_1 1/9 bushel cartons',
       'cat__Package_1 1/9 bushel crates', 'cat__Package_1/2 bushel cartons',
       'cat__Package_24 inch bins', 'cat__Package_36 inch bins',
       'cat__Package_bins', 'cat__Package_bushel cartons',
       'cat__Variety_BIG MACK TYPE', 'cat__Variety_BLUE TYPE',
       'cat__Variety_CINDERELLA', 'cat__Variety_FAIRYTALE',
       'cat__Variety_HOWDEN TYPE', 'cat__Variety_HOWDEN WHITE TYPE',
       'cat__Variety_KNUCKLE HEAD', 'cat__Variety_MINIATURE',
       'cat__Variety_PIE TYPE', 'cat__Origin_ALABAMA',
       'cat__Origin_CALIFORNIA

,ord__Item Size,cat__City Name_ATLANTA,cat__City Name_BALTIMORE,cat__City Name_BOSTON,cat__City Name_CHICAGO,cat__City Name_COLUMBIA,cat__City Name_DALLAS,cat__City Name_DETROIT,cat__City Name_LOS ANGELES,cat__City Name_MIAMI,...,cat__Origin_NEW JERSEY,cat__Origin_NEW YORK,cat__Origin_NORTH CAROLINA,cat__Origin_OHIO,cat__Origin_PENNSYLVANIA,cat__Origin_TENNESSEE,cat__Origin_TEXAS,cat__Origin_VERMONT,cat__Origin_VIRGINIA,Color
2,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0
4,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
5,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
6,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [98]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report 
from sklearn.linear_model import LogisticRegression

X = encoded_pumpkins.drop(columns=['Color'])
y = encoded_pumpkins['Color']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

model = LogisticRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(classification_report(y_test, predictions, target_names=label_encoder.classes_))
print('F1-score: ', f1_score(y_test, predictions))

              precision    recall  f1-score   support

      ORANGE       0.94      0.98      0.96       166
       WHITE       0.85      0.67      0.75        33

    accuracy                           0.92       199
   macro avg       0.89      0.82      0.85       199
weighted avg       0.92      0.92      0.92       199

F1-score:  0.7457627118644068


In [99]:
from pumpkin_input import COLUMNS, build_input_row

ALLOWED_VALUES = {
    key: sorted(pumpkins[col].unique().tolist())
    for key, col in COLUMNS.items()
}

def pumpkin_input_encoder(**kwargs):
    return ct.transform(build_input_row(kwargs, ALLOWED_VALUES))

pumpkin_input_encoder(item_size='sml', city_name='BALTIMORE', package='24 inch bins', variety='HOWDEN TYPE', origin='NEW YORK')

,ord__Item Size,cat__City Name_ATLANTA,cat__City Name_BALTIMORE,cat__City Name_BOSTON,cat__City Name_CHICAGO,cat__City Name_COLUMBIA,cat__City Name_DALLAS,cat__City Name_DETROIT,cat__City Name_LOS ANGELES,cat__City Name_MIAMI,...,cat__Origin_MICHIGAN,cat__Origin_NEW JERSEY,cat__Origin_NEW YORK,cat__Origin_NORTH CAROLINA,cat__Origin_OHIO,cat__Origin_PENNSYLVANIA,cat__Origin_TENNESSEE,cat__Origin_TEXAS,cat__Origin_VERMONT,cat__Origin_VIRGINIA
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [100]:
import pickle

artifact = {
    'preprocessor': ct,
    'model': model,
    'classes': label_encoder.classes_.tolist(),
    'allowed': ALLOWED_VALUES
}

pickle.dump(artifact, open('pumpkin-model.pkl','wb'))

loaded = pickle.load(open('pumpkin-model.pkl','rb'))
print(type(loaded))              # <class 'dict'>
print(list(loaded))              # ['preprocessor', 'model', 'classes', 'allowed']
print(type(loaded['model']))


print(model.predict(pumpkin_input_encoder(item_size='sml', city_name='BALTIMORE', package='24 inch bins', variety='HOWDEN TYPE', origin='NEW YORK')))

<class 'dict'>
['preprocessor', 'model', 'classes', 'allowed']
<class 'sklearn.linear_model._logistic.LogisticRegression'>
[0]
